<a href="https://colab.research.google.com/github/JeyasuriyaJ/jeyasuriya-codeboosters-2026/blob/main/Day-4/Day_4_BigData_PytSpark_Arcihtecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark --quiet
print('PySpark installation complete!')

PySpark installation complete!


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
  .appName('Day4_BigData_Sales') \
  .config('spark.sql.adaptive.enabled','true') \
  .getOrCreate()

print(f'Spark Version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'Application : {spark.sparkContext.appName}')

Spark Version : 4.0.2
SparkSession : ACTIVE
Application : Day4_BigData_Sales


In [3]:
df_bronze = spark.read \
  .option('header','true') \
  .option('inferSchema', 'true') \
  .csv('large_sales_data.csv')

print('===Bronze Layer - Raw Data===')
print(f'Rows:{df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
print(f'Names:{df_bronze.columns}')
df_bronze.printSchema()

===Bronze Layer - Raw Data===
Rows:5000
Columns:13
Names:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [4]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [5]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) /1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath, f))
  return total /1024

csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - (parquet_size / csv_size)) * 100

print(f'CSV Size: {csv_size:.1f} KB')
print(f'Parquet Size: {parquet_size:.1f} KB')
print(f'Reduction: {reduction:.1f}%')
print(f'\nAt 1 TB  scale: csv=1000 GB -> Parquet= {1000 * (parquet_size / 100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet
CSV Size: 529.3 KB
Parquet Size: 55.1 KB
Reduction: 89.6%

At 1 TB  scale: csv=1000 GB -> Parquet= 551 GB


In [10]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

# Correctly add a new column 'unit_price_revenue_sum' that is the sum of 'unit_price' and 'revenue'
df_bronze = df_bronze.withColumn('unit_price_revenue_sum', F.col('unit_price') + F.col('revenue'))

print('\nFirst 5 rows with new sum column:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-------------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|unit_price_quantity|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-------------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |22012              |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |12010              |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |810        

In [15]:
df_bronze.select("product","revenue","city").show(5)
df_bronze.filter(F.col("revenue") > 100000).show()

+----------+-------+---------+
|   product|revenue|     city|
+----------+-------+---------+
|   Monitor| 264000|   Mumbai|
|   Printer| 120000|    Delhi|
|     Mouse|   8000|Ahmedabad|
|    Tablet| 160000|    Surat|
|Headphones|  14000|Bangalore|
+----------+-------+---------+
only showing top 5 rows
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-------------------+----------------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|unit_price_quantity|unit_price_revenue_sum|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-------------------+----------------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Del

In [17]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

print(f'Silver layer rowsL {df_silver.count()}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(8)

Silver layer rowsL 5000
New Columns added: order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|Headphones|  38500|      2023|          7|          Medium|
|    Tablet| 128000|      2023|          7|            High|
|     Mouse|   3200|      2023|          2|             Low|
|   USB Hub|   3000|      2023|          3|             Low|
|    Tablet| 192000|      2023|          4|            High|
|Headphones|  21000|      2023|         11|          Medium|
|     Mouse|   1600|      2023|          8|             Low|
|Headphones|  45500|      2023|         10|            High|
+----------+-------+----------+-----------+----------------+
only showing top 8 rows


In [23]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

bronze_rows_count = df_bronze.count()
silver_rows_count = df_silver.count()
dropped_rows = bronze_rows_count - silver_rows_count

print(f'Initial rows in Bronze layer: {bronze_rows_count}')
print(f'Silver layer rows: {silver_rows_count}')
print(f'Number of items (rows) dropped: {dropped_rows}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(8)

print('\nProduct names and number of orders:')
df_silver.groupBy('product').agg(F.countDistinct('order_id').alias('number_of_orders')).show()

Initial rows in Bronze layer: 5000
Silver layer rows: 5000
Number of items (rows) dropped: 0
New Columns added: order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|Headphones|  38500|      2023|          7|          Medium|
|    Tablet| 128000|      2023|          7|            High|
|     Mouse|   3200|      2023|          2|             Low|
|   USB Hub|   3000|      2023|          3|             Low|
|    Tablet| 192000|      2023|          4|            High|
|Headphones|  21000|      2023|         11|          Medium|
|     Mouse|   1600|      2023|          8|             Low|
|Headphones|  45500|      2023|         10|            High|
+----------+-------+----------+-----------+----------------+
only showing top 8 rows

Product names and number of orders:
+----------+----------------+
|   product|number_of_ord

In [ ]:
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')

print('Silver Parquet saved:sales_silver.parquet')
print(f'Silver size:{get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows:{df_verify.count()} (should match Silver count)')
df_verify.printSchema()

Silver Parquet saved:sales_silver.parquet
Silver size:70.1 KB
Read-back rows:5000 (should match Silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- unit+revenue: integer (nullable = true)
 |-- unit_price_revenue_sum: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

